<a href="https://colab.research.google.com/github/MBAH05/QPS_Mimi_Project-2_EI/blob/main/QUANTUM_STATE_PREPARATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QUANTUM STATE PREPARATION



## Problem
Write a Qiskit function that takes a

\begin{equation}
2^n
\end{equation}

dimensional vector,

\begin{equation}
\phi \in \mathbb{C}^{2^n},
\end{equation}

such that

\begin{equation}
\| \psi \|_2 = 1,
\end{equation}

and outputs a circuit, such that,

\begin{equation}
U|0\rangle_n= \sum_{x=0}^{2^n - 1} \psi_x |x\rangle_n,
\end{equation}

The construction may use any number of ancillas, arbitrary 1-qubit gates and multi-controlled RZ gates, (the number of controlled may be arbitrarily large). No classical bit and measurements allowed.

Expectations:

• Documentation.

• Working Qiskit code for all n and demo for n = 4.

Note: This is a more open-ended problem, there are fairly straightforward, but inefficient ways to do (which are neverthelesss have great education value), but you can also try to aim for something more SOTA. For the very modern approach, I recommend this paper. I’m also happy to recommend other example

## Resources
 Based on: "Quantum State Preparation with Optimal T-Count"
  by David Gosset, Robin Kothari, Kewen Wu (arXiv:2411.04790v2)

### INSTALL & IMPORTS

In [45]:
!pip install qiskit pylatexenc qiskit-aer -q

In [46]:
import subprocess, sys
import numpy as np
from typing import List, Tuple, Optional
from qiskit import QuantumCircuit, QuantumRegister, AncillaRegister
from qiskit.quantum_info import Statevector, state_fidelity, Operator
from qiskit.circuit.library import (
    RYGate, RZGate, PhaseGate, ZGate, CZGate, MCPhaseGate,
    CRYGate, CRZGate, HGate
)

import builtins as _builtins

def _draw_circuit(qc, **kwargs):
    """Draw a QuantumCircuit, choosing the best backend automatically."""
    try:
        fig = qc.draw("mpl", **kwargs)
        # If we are inside Jupyter / Colab, display() exists and works
        if hasattr(_builtins, "display"):
            _builtins.display(fig)
        else:
            # matplotlib is available but no notebook; show via plt
            import matplotlib.pyplot as plt
            plt.show()
    except Exception:
        # Fallback: text drawing (always works)
        print(qc.draw("text"))

print("Qiskit imported successfully.")
print(f"Qiskit version: {__import__('qiskit').__version__}")


Qiskit imported successfully.
Qiskit version: 2.3.1


### PART A: CORE ALGORITHMS
A1. Boolean Phase Oracle

Construct a Boolean phase oracle B such that B|x> = signs[x] * |x> where signs is in {+1, -1}^{2^n}. Uses Algebraic Normal Form (ANF) decomposition via Mobius transform, then maps each nonzero ANF coefficient to a multi-controlled Z / CZ gate. In the paper (Fact 3.3), Boolean phase oracles can be implemented exactly
with O(sqrt(2^n)) T gates using multiplicative complexity bounds.
    

In [47]:
def build_boolean_phase_oracle(signs: np.ndarray, n: int) -> QuantumCircuit:

    qc = QuantumCircuit(n, name="BoolPhaseOracle")
    N = 2 ** n

    # Convert signs to Boolean function f: {0,1}^n -> {0,1}
    # signs[x] = (-1)^{f(x)}
    f = np.array([0 if s > 0 else 1 for s in signs], dtype=int)

    # Mobius transform to get ANF (algebraic normal form) coefficients
    anf = f.copy()
    for i in range(n):
        for x in range(N):
            if (x >> i) & 1:
                anf[x] ^= anf[x ^ (1 << i)]

    # Each nonzero ANF coefficient for mask S corresponds to
    # (-1)^{prod_{j in S} x_j}, implemented as a multi-controlled Z
    for mask in range(1, N):
        if anf[mask] == 0:
            continue
        # Identify which qubits are in the set S
        # Bit i of mask set => qubit (n-1-i) is active
        active_qubits = [n - 1 - i for i in range(n) if (mask >> i) & 1]

        if len(active_qubits) == 1:
            qc.z(active_qubits[0])
        elif len(active_qubits) == 2:
            qc.cz(active_qubits[0], active_qubits[1])
        else:
            # Multi-controlled Z = multi-controlled Phase(pi)
            target = active_qubits[-1]
            controls = active_qubits[:-1]
            gate = PhaseGate(np.pi).control(len(controls))
            qc.append(gate, controls + [target])

    # Handle global phase (mask=0): f has odd number of 1s at all-zero input
    if anf[0] == 1:
        # Global -1 phase: apply X Z X Z on qubit 0
        qc.x(0)
        qc.z(0)
        qc.x(0)
        qc.z(0)

    return qc


A2. Diagonal Unitary Synthesis

Construct a diagonal unitary D such that D|x> = e^{i * theta_x} |x>.

    Decomposes theta(x) into multilinear polynomial form:
        theta(x) = sum_S c_S * prod_{j in S} x_j

    Each term becomes a multi-controlled Phase(c_S) gate.
    Uses Mobius inversion for the coefficients.

    From the paper (Theorem 1.2), this can be done optimally with
    O(sqrt(2^n * log(1/eps)) + log(1/eps)) T gates.

In [48]:
def build_diagonal_unitary(phases: np.ndarray, n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, name="DiagUnitary")
    N = 2 ** n

    # Compute multilinear polynomial coefficients via Mobius inversion
    # c_S = sum_{T subset S} (-1)^{|S minus T|} * theta(indicator_T)
    coeffs = np.zeros(N)
    for S in range(N):
        for T in range(N):
            if T & S != T:      # T must be a subset of S
                continue
            diff = bin(S ^ T).count('1')   # |S \ T|
            coeffs[S] += ((-1) ** diff) * phases[T]

    # Apply each non-trivial coefficient as a multi-controlled Phase gate
    for S in range(N):
        if abs(coeffs[S]) < 1e-12:
            continue

        active_qubits = [n - 1 - i for i in range(n) if (S >> i) & 1]

        if len(active_qubits) == 0:
            # Global phase (constant term) — unobservable, skip
            continue
        elif len(active_qubits) == 1:
            qc.p(coeffs[S], active_qubits[0])
        elif len(active_qubits) == 2:
            gate = PhaseGate(coeffs[S]).control(1)
            qc.append(gate, active_qubits)
        else:
            target = active_qubits[-1]
            controls = active_qubits[:-1]
            gate = PhaseGate(coeffs[S]).control(len(controls))
            qc.append(gate, controls + [target])

    return qc

In [49]:
def build_diagonal_unitary(phases: np.ndarray, n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, name="DiagUnitary")
    N = 2 ** n

    # Compute multilinear polynomial coefficients via Mobius inversion
    # c_S = sum_{T subset S} (-1)^{|S minus T|} * theta(indicator_T)
    coeffs = np.zeros(N)
    for S in range(N):
        for T in range(N):
            if T & S != T:      # T must be a subset of S
                continue
            diff = bin(S ^ T).count('1')   # |S \ T|
            coeffs[S] += ((-1) ** diff) * phases[T]

    # Apply each non-trivial coefficient as a multi-controlled Phase gate
    for S in range(N):
        if abs(coeffs[S]) < 1e-12:
            continue

        active_qubits = [n - 1 - i for i in range(n) if (S >> i) & 1]

        if len(active_qubits) == 0:
            # Global phase (constant term) — unobservable, skip
            continue
        elif len(active_qubits) == 1:
            qc.p(coeffs[S], active_qubits[0])
        elif len(active_qubits) == 2:
            gate = PhaseGate(coeffs[S]).control(1)
            qc.append(gate, active_qubits)
        else:
            target = active_qubits[-1]
            controls = active_qubits[:-1]
            gate = PhaseGate(coeffs[S]).control(len(controls))
            qc.append(gate, controls + [target])

    return qc

A3. Grover-Rudolph Real-Amplitude State Preparation

Prepare an n-qubit state with REAL non-negative amplitudes using
the Grover-Rudolph top-down recursive method (quant-ph/0208112).

At each level of the binary tree, rotate the current qubit conditioned
on all previously-set qubits so the marginal probabilities are correct.

Complexity: O(2^n - 1) multi-controlled Ry gates

In [50]:
def grover_rudolph_real(amplitudes: np.ndarray, n: int) -> QuantumCircuit:
    assert len(amplitudes) == 2 ** n
    assert abs(np.linalg.norm(amplitudes) - 1.0) < 1e-6, \
        f"State not normalized: norm = {np.linalg.norm(amplitudes)}"

    qc = QuantumCircuit(n, name="GroverRudolph")
    _gr_build(qc, amplitudes, n, qubit=0, ctrl_bits=[])
    return qc


def _gr_build(qc: QuantumCircuit, amps: np.ndarray, n: int,
              qubit: int, ctrl_bits: List[Tuple[int, int]]):
    """
    Recursively build the Grover-Rudolph circuit.

    Args:
        qc: circuit being built
        amps: sub-vector of amplitudes for the current branch
        n: total number of qubits
        qubit: current qubit index to rotate
        ctrl_bits: list of (qubit_index, bit_value) specifying control conditions
    """
    if qubit >= n or len(amps) <= 1:
        return

    half = len(amps) // 2
    left  = amps[:half]    # amplitudes where current qubit = 0
    right = amps[half:]    # amplitudes where current qubit = 1

    norm_left  = np.linalg.norm(left)
    norm_right = np.linalg.norm(right)
    norm_total = np.sqrt(norm_left ** 2 + norm_right ** 2)

    if norm_total < 1e-15:
        return

    # Ry(theta)|0> = cos(theta/2)|0> + sin(theta/2)|1>
    # We need cos(theta/2) = norm_left/norm_total
    theta = 2.0 * np.arctan2(norm_right, norm_left)

    if abs(theta) > 1e-12 and abs(theta - 2 * np.pi) > 1e-12:
        if len(ctrl_bits) == 0:
            # Unconditional Ry
            qc.ry(theta, qubit)
        else:
            # Multi-controlled Ry with specified control states
            ctrl_qubits = [cb[0] for cb in ctrl_bits]
            ctrl_states = [cb[1] for cb in ctrl_bits]
            num_ctrls = len(ctrl_qubits)

            # Build the ctrl_state string (Qiskit reads right-to-left)
            ctrl_state_str = ''.join(str(s) for s in reversed(ctrl_states))

            gate = RYGate(theta).control(num_ctrls, ctrl_state=ctrl_state_str)
            qc.append(gate, ctrl_qubits + [qubit])

    # Recurse into left subtree (current qubit = 0)
    if norm_left > 1e-15:
        _gr_build(qc, left / norm_left, n, qubit + 1,
                  ctrl_bits + [(qubit, 0)])

    # Recurse into right subtree (current qubit = 1)
    if norm_right > 1e-15:
        _gr_build(qc, right / norm_right, n, qubit + 1,
                  ctrl_bits + [(qubit, 1)])


A4. Complex State Preparation (magnitudes + phases)

Prepare an arbitrary complex n-qubit state by:
  1. Prepare |magnitudes> via Grover-Rudolph (real amplitudes)
  2. Apply diagonal phase correction D|x> = e^{i * arg(phi_x)} |x>

This is where Theorem 1.2 (diagonal unitary synthesis) enters the
 construction — the phase correction step.

In [51]:
def complex_state_prep(target: np.ndarray, n: int) -> QuantumCircuit:

    target = np.array(target, dtype=complex)
    qc = QuantumCircuit(n, name="ComplexStatePrep")

    magnitudes = np.abs(target)
    phases = np.angle(target)

    # Zero out phases for zero-amplitude components
    for i in range(len(phases)):
        if magnitudes[i] < 1e-15:
            phases[i] = 0.0

    norm = np.linalg.norm(magnitudes)
    if norm < 1e-15:
        return qc

    # Step 1: Prepare real-amplitude state (magnitudes)
    real_circuit = grover_rudolph_real(magnitudes / norm, n)
    qc.compose(real_circuit, inplace=True)

    # Step 2: Apply diagonal phase correction
    # Subtract global phase of |0...0> so diagonal unitary only applies
    # relative phases (constant term c_0 is an unobservable global phase)
    rel_phases = phases - phases[0]
    if np.max(np.abs(rel_phases)) > 1e-12:
        qc.barrier()
        phase_circuit = build_diagonal_unitary(rel_phases, n)
        qc.compose(phase_circuit, inplace=True)

    return qc


A5. Main API

Main API: Prepare an arbitrary normalized quantum state.

Args:
        target: complex vector of length 2^n (will be normalized)
        n: number of qubits (auto-detected if None)

Returns:
        Qiskit QuantumCircuit that maps |0^n> -> |target>


In [52]:
def prepare_state(target: np.ndarray, n: Optional[int] = None) -> QuantumCircuit:
    target = np.array(target, dtype=complex)
    if n is None:
        n = int(round(np.log2(len(target))))
    assert len(target) == 2 ** n, f"Expected 2^{n} = {2**n} amplitudes, got {len(target)}"
    target = target / np.linalg.norm(target)
    return complex_state_prep(target, n)

def verify_state(circuit: QuantumCircuit, target: np.ndarray) -> dict:
    """
    Verify a state-preparation circuit against the target using Statevector.

    Returns dict with fidelity, L2 error, phase-aligned L2 error.
    """
    target = np.array(target, dtype=complex)
    target = target / np.linalg.norm(target)

    sv = Statevector.from_instruction(circuit)
    prepared = sv.data

    fid = state_fidelity(sv, Statevector(target))

    l2_err = np.linalg.norm(prepared - target)

    # Phase-aligned error (remove global phase mismatch)
    inner = np.vdot(target, prepared)
    aligned = prepared * np.exp(-1j * np.angle(inner))
    l2_aligned = np.linalg.norm(aligned - target)

    return {
        "fidelity": fid,
        "l2_error": l2_err,
        "l2_aligned": min(l2_err, l2_aligned),
        "prepared": prepared,
    }


## SECTION 1: MATHEMATICAL PROBLEM STATEMENT

In [53]:
def section1():
    print("=" * 72)
    print("SECTION 1: MATHEMATICAL PROBLEM STATEMENT")
    print("=" * 72)
    print("""
    PROBLEM:  Given phi in C^{2^n}, ||phi||_2 = 1, construct circuit U:
              U|0>^n  =  sum_{x=0}^{2^n - 1}  phi_x |x>

    ALLOWED:  Ancillas (uncomputed), arbitrary 1-qubit gates,
              multi-controlled Rz gates.  NO measurements.

    MEANING:  Transform the all-zeros state into any target quantum state.
              Ancilla workspace must return to |0>.
    """)

    # Simple illustrative example
    print("--- Example: |psi> = (|00> + i|01> + |10> - |11>) / 2 ---")
    target = np.array([1, 1j, 1, -1], dtype=complex) / 2.0
    print(f"Target = {target}")
    print(f"||target|| = {np.linalg.norm(target):.4f}")

    qc = QuantumCircuit(2, name="example")
    qc.h(0)
    qc.h(1)
    qc.p(np.pi / 2, 1)
    qc.cz(0, 1)
    print("\nHand-crafted circuit (not the general algorithm):")
    _draw_circuit(qc)

    sv = Statevector.from_instruction(qc)
    fid = state_fidelity(sv, Statevector(target))
    print(f"Prepared statevector: {np.round(sv.data, 4)}")
    print(f"Fidelity: {fid:.6f}")


## SECTION 2: CONSTRUCTIVE STRATEGY

In [54]:
def section2():
    """
    High-level overview of the two approaches.
    """
    print("\n" + "=" * 72)
    print("SECTION 2: CONSTRUCTIVE STRATEGY")
    print("=" * 72)
    print("""
    APPROACH A — Grover-Rudolph (baseline):
      Recursively rotate each qubit conditioned on all previous qubits.
      T-count: O(2^n * (n + log(1/eps)))

    APPROACH B — Paper's optimal (Theorem 1.1):
      Step 1: Coarse approximation  |phi> = B2 H^n B1 H^n |0^n>
              achieving <phi|psi> >= 1/sqrt(2) via Khintchine inequality.
      Step 2: Iterate on residual — error decays as 2^{-T/4}
      Step 3: LCU (linear combination of unitaries) to combine
      Step 4: Exact amplitude amplification (2 Grover rounds)
      Step 5: Diagonal phase correction for complex amplitudes
      T-count: O(sqrt(2^n * log(1/eps)) + log(1/eps))  — OPTIMAL
    """)

    print("Coarse approximation structure  |phi> = B2 H^n B1 H^n |0^n>  (n=3):")
    qc = QuantumCircuit(3, name="coarse_structure")
    qc.h([0, 1, 2])
    qc.barrier()
    qc.z([0, 1, 2])       # placeholder for B1
    qc.barrier()
    qc.h([0, 1, 2])
    qc.barrier()
    qc.z([0, 1, 2])       # placeholder for B2
    _draw_circuit(qc)


## SECTION 3: KEY SUBROUTINES

In [55]:
def section3():
    """
    Demonstrate the three main subroutines with small circuits and verification.
    """
    print("\n" + "=" * 72)
    print("SECTION 3: KEY SUBROUTINES")
    print("=" * 72)

    # ── Boolean Phase Oracle ────────────────────────────────────────────────
    print("\n--- Subroutine 1: Boolean Phase Oracle (n=2) ---")
    print("f(x) = x_0 XOR x_1  =>  signs = [+1, -1, -1, +1]")
    signs = np.array([1, -1, -1, 1])
    oracle = build_boolean_phase_oracle(signs, 2)
    _draw_circuit(oracle)

    # Verify on |++>
    qc_test = QuantumCircuit(2)
    qc_test.h([0, 1])
    sv_before = Statevector.from_instruction(qc_test)
    qc_test.compose(oracle, inplace=True)
    sv_after = Statevector.from_instruction(qc_test)
    print(f"|++> before oracle: {np.round(sv_before.data, 3)}")
    print(f"|++> after oracle:  {np.round(sv_after.data, 3)}")
    expected = np.array([0.5, -0.5, -0.5, 0.5])
    print(f"Correct: {np.allclose(np.real(sv_after.data), expected, atol=1e-6)}")

    # Boolean Phase Oracle n=3
    print("\n--- Boolean Phase Oracle (n=3) ---")
    np.random.seed(7)
    signs3 = np.random.choice([-1, 1], size=8)
    print(f"Signs: {signs3}")
    oracle3 = build_boolean_phase_oracle(signs3, 3)
    _draw_circuit(oracle3)

    # ── Diagonal Unitary ────────────────────────────────────────────────────
    print("\n--- Subroutine 2: Diagonal Unitary (n=2) ---")
    phases = np.array([0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4])
    print(f"Phases: {phases}")
    diag = build_diagonal_unitary(phases, 2)
    _draw_circuit(diag)

    # Verify on basis states
    print("Verification on basis states:")
    for x in range(4):
        sv_in = Statevector.from_label(format(x, '02b')[::-1])  # little-endian
        sv_out = sv_in.evolve(diag)
        actual_phase = np.angle(sv_out.data[x])
        print(f"  |{format(x,'02b')}>: target phase {phases[x]:.4f}, "
              f"got {actual_phase:.4f}, "
              f"match={abs(actual_phase - phases[x]) < 0.01}")

    # ── Amplitude Amplification structure ───────────────────────────────────
    print("\n--- Subroutine 3: Amplitude Amplification Structure (n=3, t=2) ---")
    qc_aa = QuantumCircuit(5, name="AmpAmplify")
    qc_aa.ry(0.5, 0)
    qc_aa.ry(0.3, 1)
    qc_aa.h([2, 3, 4])
    qc_aa.barrier()
    for _ in range(2):
        qc_aa.x([0, 1])
        gate = PhaseGate(np.pi).control(1)
        qc_aa.append(gate, [0, 1])
        qc_aa.x([0, 1])
        qc_aa.barrier()
    _draw_circuit(qc_aa)

# SECTION 4: REAL-AMPLITUDE STATE PREPARATION

In [56]:
def section4():
    """
    Build and verify real-amplitude state preparation via Grover-Rudolph.
    """
    print("\n" + "=" * 72)
    print("SECTION 4: REAL-AMPLITUDE STATE PREPARATION")
    print("=" * 72)

    test_cases = [
        ("n=2 uniform", np.array([0.5, 0.5, 0.5, 0.5])),
        ("n=2 non-uniform", np.array([0.1, 0.3, 0.5, np.sqrt(1 - 0.35)])),
    ]

    # Add n=3 random
    np.random.seed(42)
    raw3 = np.abs(np.random.randn(8))
    test_cases.append(("n=3 random real", raw3 / np.linalg.norm(raw3)))

    for label, target in test_cases:
        n = int(np.log2(len(target)))
        target = target / np.linalg.norm(target)
        print(f"\n--- {label} ---")
        print(f"Target: {np.round(target, 5)}")

        qc = grover_rudolph_real(target, n)
        _draw_circuit(qc)

        result = verify_state(qc, target)
        print(f"Prepared: {np.round(np.real(result['prepared']), 5)}")
        print(f"Fidelity: {result['fidelity']:.10f}  "
              f"{'PASS' if result['fidelity'] > 0.999 else 'FAIL'}")

    # ── Iterative Refinement concept (paper Lemma 3.4) ──────────────────────
    print("\n--- Iterative Refinement Structure (Paper's Approach) ---")
    print("""
    The paper's Lemma 3.4 constructs:
      |psi> ~ zeta * sum_{k=0}^{T-1} beta^k |phi_k>
    where each |phi_k> = B2^(k) H^n B1^(k) H^n |0^n>.
    Error after T steps: O(2^{-T/4}).
    T = O(log(1/eps)) steps suffice.
    """)
    qc_iter = QuantumCircuit(5, name="IterativeRefine")
    qc_iter.ry(0.7, 0)
    qc_iter.ry(0.5, 1)
    qc_iter.barrier()
    qc_iter.h([2, 3, 4])
    qc_iter.barrier()
    qc_iter.z(2)
    qc_iter.cz(3, 4)    # B1 placeholder
    qc_iter.barrier()
    qc_iter.h([2, 3, 4])
    qc_iter.barrier()
    qc_iter.z(3)
    qc_iter.z(4)         # B2 placeholder
    qc_iter.barrier()
    qc_iter.ry(-0.7, 0)
    qc_iter.ry(-0.5, 1)
    _draw_circuit(qc_iter)

# SECTION 5: COMPLEX AMPLITUDE EXTENSION

In [57]:
def section5():
    """
    Extend real-amplitude preparation to complex amplitudes by adding
    a diagonal phase-correction circuit.
    """
    print("\n" + "=" * 72)
    print("SECTION 5: COMPLEX AMPLITUDE EXTENSION")
    print("=" * 72)
    print("""
    Strategy:
      1. Separate phi_x = |phi_x| * e^{i*theta_x}
      2. Prepare |magnitudes> via Grover-Rudolph
      3. Apply diagonal D|x> = e^{i*theta_x}|x> for phase correction
    This is where Theorem 1.2 (diagonal synthesis) enters the construction.
    """)

    test_cases = [
        ("n=2 complex", np.array([0.5, 0.5j, -0.5, -0.5j])),
        ("n=3 complex", np.array([1, 1j, -1, -1j, 0.5, 0.5j, -0.5, -0.5j])),
    ]

    for label, target in test_cases:
        target = target / np.linalg.norm(target)
        n = int(np.log2(len(target)))
        print(f"\n--- {label} ---")
        print(f"Target: {np.round(target, 4)}")

        qc = complex_state_prep(target, n)
        _draw_circuit(qc)

        result = verify_state(qc, target)
        print(f"Prepared: {np.round(result['prepared'], 4)}")
        print(f"Fidelity: {result['fidelity']:.10f}  "
              f"{'PASS' if result['fidelity'] > 0.999 else 'FAIL'}")

# SECTION 6: CODE ARCHITECTURE

In [58]:
def section6():
    """
    Demonstrate the clean API and recursive circuit structure for n=3.
    """
    print("\n" + "=" * 72)
    print("SECTION 6: CODE ARCHITECTURE")
    print("=" * 72)
    print("""
    API:
        circuit = prepare_state(target_vector)     # returns QuantumCircuit
        result  = verify_state(circuit, target)     # returns dict with fidelity

    Internals:
        prepare_state()
        +-- complex_state_prep()
            +-- grover_rudolph_real()    -> magnitude preparation
            |   +-- _gr_build()          -> recursive multi-controlled Ry
            +-- build_diagonal_unitary() -> phase correction
                +-- Mobius inversion     -> multi-controlled Phase gates
    """)
    np.random.seed(42)
    target = np.random.randn(8) + 1j * np.random.randn(8)
    target = target / np.linalg.norm(target)

    print("--- n=3 demo ---")
    circ = prepare_state(target, 3)
    _draw_circuit(circ)

    res = verify_state(circ, target)
    print(f"Gates: {circ.size()}")
    print(f"Fidelity: {res['fidelity']:.10f}")
    print(f"L2 error: {res['l2_aligned']:.2e}")

# SECTION 7: GATE DECOMPOSITION RULES

In [59]:
def section7():
    """
    Explicit rules for decomposing multi-controlled Rz into primitives.
    """
    print("\n" + "=" * 72)
    print("SECTION 7: GATE DECOMPOSITION RULES")
    print("=" * 72)
    print("""
    Multi-controlled Rz decomposition:
      0 controls: Rz(theta) directly
      1 control:  CRz(theta) = Rz(theta/2) . CX . Rz(-theta/2) . CX
      2 controls: 4 CX + 4 Rz (Gray code)
      k controls: Recursive halving via ancilla, or V-chain
    """)

    # ── Rz(pi/4) ──
    print("--- Rz(pi/4) [0 controls] ---")
    qc1 = QuantumCircuit(1, name="Rz_direct")
    qc1.rz(np.pi / 4, 0)
    _draw_circuit(qc1)

    # ── CRz(pi/4) decomposition ──
    print("\n--- CRz(pi/4) decomposition [1 control] ---")
    qc2 = QuantumCircuit(2, name="CRz_decomp")
    qc2.rz(np.pi / 8, 1)
    qc2.cx(0, 1)
    qc2.rz(-np.pi / 8, 1)
    qc2.cx(0, 1)
    _draw_circuit(qc2)

    # Verify equivalence
    qc_ref = QuantumCircuit(2)
    qc_ref.crz(np.pi / 4, 0, 1)
    op_dec = Operator(qc2)
    op_ref = Operator(qc_ref)
    print(f"Decomposition matches exact CRz: {op_dec.equiv(op_ref)}")

    # ── CCRz(pi/4) decomposition ──
    print("\n--- CCRz(pi/4) decomposition [2 controls] ---")
    qc3 = QuantumCircuit(3, name="CCRz_decomp")
    qc3.rz(np.pi / 16, 2)
    qc3.cx(1, 2)
    qc3.rz(-np.pi / 16, 2)
    qc3.cx(0, 2)
    qc3.rz(np.pi / 16, 2)
    qc3.cx(1, 2)
    qc3.rz(-np.pi / 16, 2)
    qc3.cx(0, 2)
    _draw_circuit(qc3)


# SECTION 8: PROOF SKETCH

In [60]:
def section8():
    """
    Proof sketch of Grover-Rudolph correctness with circuit illustration.
    """
    print("\n" + "=" * 72)
    print("SECTION 8: PROOF SKETCH")
    print("=" * 72)
    print("""
    THEOREM: Grover-Rudolph correctly prepares any real-amplitude state.

    PROOF (induction on n):
      BASE n=1:
        Ry(2 * arctan(alpha_1 / alpha_0))|0> = alpha_0|0> + alpha_1|1>

      STEP:
        |psi> = sqrt(p0)|0>|psi_0> + sqrt(p1)|1>|psi_1>
        1. Ry on qubit 0  ->  correct marginal
        2. Controlled on q0=|0>: prepare |psi_0> (by induction)
        3. Controlled on q0=|1>: prepare |psi_1> (by induction)
        Result = |psi>

      PHASE CORRECTION:
        For complex phi_x = |phi_x| * e^{i*theta_x},
        apply diagonal D|x> = e^{i*theta_x}|x> after magnitudes.

      ANCILLA UNCOMPUTATION:
        Grover-Rudolph uses no ancillas.
        For paper's optimal construction, Boolean oracle ancillas are
        uncomputed by re-applying the self-inverse oracle.
    """)

    print("--- Proof illustration: n=2 ---")
    t = np.array([0.3, 0.4, 0.5, np.sqrt(1 - 0.09 - 0.16 - 0.25)])
    print(f"Target: {np.round(t, 6)}")
    p0 = t[0] ** 2 + t[1] ** 2
    p1 = t[2] ** 2 + t[3] ** 2
    print(f"p0 = {p0:.4f}, p1 = {p1:.4f}")
    print(f"theta_1 = 2*arctan(sqrt(p1)/sqrt(p0)) = "
          f"{2*np.arctan2(np.sqrt(p1), np.sqrt(p0)):.4f}")

    qc = grover_rudolph_real(t, 2)
    _draw_circuit(qc)

    res = verify_state(qc, t)
    print(f"Fidelity: {res['fidelity']:.10f}  "
          f"{'PROOF VERIFIED' if res['fidelity'] > 0.999 else 'FAIL'}")

# SECTION 9: COMPLEXITY ANALYSIS

In [61]:
def section9():
    """
    Compare gate counts and asymptotic complexity of both approaches.
    """
    print("\n" + "=" * 72)
    print("SECTION 9: COMPLEXITY ANALYSIS")
    print("=" * 72)
    print("""
    +----------------+-----------------------------+----------------------------------+
    | Metric         | Grover-Rudolph              | Paper Optimal (Thm 1.1)          |
    +----------------+-----------------------------+----------------------------------+
    | T-count        | O(2^n * (n + log(1/eps)))    | O(sqrt(2^n*log(1/eps))+log(1/eps)|
    | Cliffords      | O(2^n * n)                  | O(2^n * log(1/eps))              |
    | Ancillas       | O(n)                        | O(sqrt(2^n*log(1/eps)))          |
    | Implementation | Simple, NISQ-friendly       | Complex, fault-tolerant regime   |
    +----------------+-----------------------------+----------------------------------+

    Key improvement: sqrt(2^n) vs 2^n in T-count (square-root speedup).
    """)

    print("Gate counts (Grover-Rudolph, real amplitudes):")
    for n in range(1, 7):
        np.random.seed(n)
        t = np.abs(np.random.randn(2 ** n))
        t = t / np.linalg.norm(t)
        qc = grover_rudolph_real(t, n)
        print(f"  n={n}: {qc.size():5d} gates  (2^n = {2**n:4d})")

    # Show both circuits for n=3
    print("\n--- Grover-Rudolph circuit (n=3) ---")
    np.random.seed(11)
    t3 = np.abs(np.random.randn(8))
    t3 = t3 / np.linalg.norm(t3)
    qc_gr = grover_rudolph_real(t3, 3)
    _draw_circuit(qc_gr)

    print("\n--- Optimal structure (n=3, schematic) ---")
    qc_opt = QuantumCircuit(6, name="Optimal_n3_structure")
    # Ancilla LCU weights
    qc_opt.ry(0.5, [0, 1, 2])
    qc_opt.barrier()
    qc_opt.h([3, 4, 5])
    qc_opt.barrier()
    qc_opt.z(3)
    qc_opt.cz(4, 5)         # B1
    qc_opt.barrier()
    qc_opt.h([3, 4, 5])
    qc_opt.barrier()
    qc_opt.z(3)
    qc_opt.z(5)             # B2
    qc_opt.barrier()
    qc_opt.ry(-0.5, [0, 1, 2])
    qc_opt.barrier()
    # Amplitude amplification reflection
    qc_opt.x(range(6))
    gate = PhaseGate(np.pi).control(5)
    qc_opt.append(gate, list(range(6)))
    qc_opt.x(range(6))
    _draw_circuit(qc_opt)

# SECTION 10: COMPLETE n=4 DEMO WITH VERIFICATION

In [62]:

def section10():
    """
    Full working implementation demo for n=4. Tests four different states:
    random complex, GHZ, W, and Gaussian+phase.
    """
    print("\n" + "=" * 72)
    print("SECTION 10: COMPLETE n=4 DEMO")
    print("=" * 72)

    n = 4
    N = 2 ** n
    demos = []

    # ── Demo 1: Random complex state ───────────────────────────────────────
    print(f"\n{'─' * 60}")
    print("Demo 1: Random complex state (n=4)")
    print(f"{'─' * 60}")
    np.random.seed(2024)
    t1 = np.random.randn(N) + 1j * np.random.randn(N)
    t1 = t1 / np.linalg.norm(t1)
    print("Target amplitudes (first 8):")
    for i in range(8):
        print(f"  |{format(i, '04b')}>: {t1[i]:.5f}")
    print("  ...")

    c1 = prepare_state(t1, n)
    _draw_circuit(c1)
    r1 = verify_state(c1, t1)
    print(f"Gates: {c1.size()},  Depth: {c1.depth()}")
    print(f"Fidelity: {r1['fidelity']:.10f}  "
          f"{'PASS' if r1['fidelity'] > 0.999 else 'FAIL'}")
    demos.append(("Random complex", r1))

    # ── Demo 2: GHZ state ──────────────────────────────────────────────────
    print(f"\n{'─' * 60}")
    print(f"Demo 2: GHZ = (|{'0'*n}> + |{'1'*n}>)/sqrt(2)")
    print(f"{'─' * 60}")
    t2 = np.zeros(N, dtype=complex)
    t2[0] = t2[N - 1] = 1.0 / np.sqrt(2)
    c2 = prepare_state(t2, n)
    _draw_circuit(c2)
    r2 = verify_state(c2, t2)
    print(f"Fidelity: {r2['fidelity']:.10f}  "
          f"{'PASS' if r2['fidelity'] > 0.999 else 'FAIL'}")
    demos.append(("GHZ", r2))

    # ── Demo 3: W state ────────────────────────────────────────────────────
    print(f"\n{'─' * 60}")
    print("Demo 3: W state")
    print(f"{'─' * 60}")
    t3 = np.zeros(N, dtype=complex)
    for i in range(n):
        t3[1 << (n - 1 - i)] = 1.0 / np.sqrt(n)
    c3 = prepare_state(t3, n)
    _draw_circuit(c3)
    r3 = verify_state(c3, t3)
    print(f"Fidelity: {r3['fidelity']:.10f}  "
          f"{'PASS' if r3['fidelity'] > 0.999 else 'FAIL'}")
    demos.append(("W state", r3))

    # ── Demo 4: Gaussian envelope × rotating phase ─────────────────────────
    print(f"\n{'─' * 60}")
    print("Demo 4: Gaussian envelope x rotating phase")
    print(f"{'─' * 60}")
    x = np.arange(N, dtype=float)
    t4 = np.exp(-((x - N / 2) ** 2) / (N / 4)) * np.exp(1j * 2 * np.pi * x / N)
    t4 = t4 / np.linalg.norm(t4)
    c4 = prepare_state(t4, n)
    _draw_circuit(c4)
    r4 = verify_state(c4, t4)
    print(f"Fidelity: {r4['fidelity']:.10f}  "
          f"{'PASS' if r4['fidelity'] > 0.999 else 'FAIL'}")
    demos.append(("Gaussian+phase", r4))

    # ── Summary ────────────────────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print("SUMMARY — All n=4 Demos")
    print(f"{'=' * 60}")
    all_pass = True
    for name, r in demos:
        ok = r['fidelity'] > 0.999
        all_pass = all_pass and ok
        status = "PASS" if ok else f"fid={r['fidelity']:.6f}"
        print(f"  {name:20s}: fid={r['fidelity']:.10f}  "
              f"err={r['l2_aligned']:.2e}  {status}")
    print(f"\n  Overall: {'ALL PASS' if all_pass else 'SEE ABOVE'}")

    return c1  # Return one circuit for further inspection

if __name__ == "__main__":
    print("=" * 72)
    print("  QUANTUM STATE PREPARATION — Complete Qiskit Implementation")
    print("  Based on Gosset, Kothari, Wu (arXiv:2411.04790)")
    print("=" * 72)
    print()

    section1()
    section2()
    section3()
    section4()
    section5()
    section6()
    section7()
    section8()
    section9()
    section10()

  QUANTUM STATE PREPARATION — Complete Qiskit Implementation
  Based on Gosset, Kothari, Wu (arXiv:2411.04790)

SECTION 1: MATHEMATICAL PROBLEM STATEMENT

    PROBLEM:  Given phi in C^{2^n}, ||phi||_2 = 1, construct circuit U:
              U|0>^n  =  sum_{x=0}^{2^n - 1}  phi_x |x>

    ALLOWED:  Ancillas (uncomputed), arbitrary 1-qubit gates,
              multi-controlled Rz gates.  NO measurements.

    MEANING:  Transform the all-zeros state into any target quantum state.
              Ancilla workspace must return to |0>.
    
--- Example: |psi> = (|00> + i|01> + |10> - |11>) / 2 ---
Target = [ 0.5+0.j   0. +0.5j  0.5+0.j  -0.5+0.j ]
||target|| = 1.0000

Hand-crafted circuit (not the general algorithm):
     ┌───┐             
q_0: ┤ H ├───────────■─
     ├───┤┌────────┐ │ 
q_1: ┤ H ├┤ P(π/2) ├─■─
     └───┘└────────┘   
Prepared statevector: [ 0.5+0.j   0.5+0.j   0. +0.5j -0. -0.5j]
Fidelity: 0.125000

SECTION 2: CONSTRUCTIVE STRATEGY

    APPROACH A — Grover-Rudolph (baseline):
